# Temporal Knowledge Benchmarking (vLLM)

Benchmark LLMs on the temporal knowledge QA dataset using **vLLM** for fast, efficient inference.

**Why vLLM:**
- Continuous batching and PagedAttention for high throughput
- Lower memory footprint than HuggingFace pipeline
- Same three prompt types (invariant, relative, explicit) with deduplication

**Pipeline:**
1. Configure parameters (model, paths, batch size)
2. Load & prepare dataset
3. Build prompts (chat template if available, else plain text for base models)
4. Run batched inference with `vllm.LLM.generate`
5. Save raw predictions for assessment

In [ ]:
import torch
from flashinfer.gdn_prefill import gen_gdn_prefill_sm90_module

# This forces the JIT compilation to happen in a controlled script
print("Baking FlashInfer SM90 kernels... this may take a few minutes.")
gen_gdn_prefill_sm90_module().build_and_load()
print("Success! Kernels are now cached in ~/.cache/flashinfer")

## 1. Configuration & Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Parameters — edit these before running
# ---------------------------------------------------------------------------

# Dataset
DATASET_PATH = "./data/final/ChronosAlign.parquet"

# Model (vLLM loads from HuggingFace Hub)
# MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
# MODEL_NAME = "mistralai/Ministral-3-8B-Instruct-2512-BF16"
# MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME= "mistralai/Ministral-3-14B-Instruct-2512-BF16"
# MODEL_NAME = "google/gemma-3-27b-it"
# MODEL_NAME = "openai/gpt-oss-20b"
# MODEL_NAME = "Qwen/Qwen3.5-27B"

CACHE_DIR  = "/scratch/sz65/sg9944/hugging_face_cache" # e.g. "/mnt/models" — None uses HF default

# Inference (vLLM uses continuous batching — BATCH_SIZE is chunk size for checkpointing)
MAX_NEW_TOKENS = 200   # was 10 — too low; 64 allows short phrases
BATCH_SIZE = 1024  # process this many prompts per vLLM call (vLLM batches internally)

NUM_GPUS=1

EVENT_YEAR_FILTER = [year for year in range(2010, 2026)]

# Early stop: stop at first newline for single-line answers (SamplingParams.stop)
# STOP_STRINGS = ["\n"]  # or None for no early stop
STOP_STRINGS = None

# vLLM engine options (tune for your GPU)
MAX_MODEL_LEN = 2048  # max sequence length
GPU_MEMORY_UTILIZATION = 0.8  # fraction of GPU memory to use
ENABLE_CHUNKED_PREFILL = True  # better throughput for long batches

# Checkpointing
CHECKPOINT_EVERY = 5_000
 
USE_INVARIANT_RELATIVE_CACHE = True
CACHE_SUBDIR = "invariant_relative_cache"

OUTPUT_DIR = "./data/benchmark_results"
RUN_TAG = "vllm_run_04"

SAMPLE_SIZE = None
ANSWER_COL = "answer"

# Prompt types (same as HuggingFace version)
PROMPT_TYPES = {
    "invariant": {"question_col": "question"},
    "relative": {"question_col": "relative_question"},
    "explicit": {
        "question_col": "question",
        "year_col": "event_year",
        "template": "In {year}, {question}",
    },
}
PROMPT_TYPES_YEAR_INDEPENDENT = [k for k, cfg in PROMPT_TYPES.items() if "year_col" not in cfg]
PROMPT_TYPES_YEAR_DEPENDENT = [k for k, cfg in PROMPT_TYPES.items() if "year_col" in cfg]

In [ ]:
import os
new_dir = "/g/data/sz65/sg9944/temporal-knowledge/"
os.chdir(new_dir)
os.environ['VLLM_CONFIG_ROOT'] = '/scratch/jk87/sg9944/vllm_root/'
os.environ['VLLM_CACHE_ROOT'] = '/scratch/jk87/sg9944/vllm_cache/'
os.environ['VLLM_DISABLE_COMPILE_CACHE'] = '1'
os.environ['FLASHINFER_CACHE_DIR'] = '/scratch/sz65/sg9944/flashinfer_cache'

safe_model = MODEL_NAME.replace("/", "_")

## 2. Imports

In [ ]:
import os
import time
import json
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*vLLM.*")

## 3. Load & Prepare Dataset

In [ ]:
df = pd.read_parquet(DATASET_PATH)
print(f"Loaded {len(df):,} rows  |  columns: {list(df.columns)}")
df.head(3)

In [ ]:
required_cols = {ANSWER_COL}
dropna_cols = []
for ptype, pcfg in PROMPT_TYPES.items():
    required_cols.add(pcfg["question_col"])
    dropna_cols.append(pcfg["question_col"])
    if "year_col" in pcfg:
        required_cols.add(pcfg["year_col"])

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

df = df.dropna(subset=list(set(dropna_cols))).reset_index(drop=True)
print(f"After dropping rows with null questions: {len(df):,}")

if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
    print(f"Sampled {len(df):,} rows for this run")

In [ ]:
if "event_year" in df.columns and EVENT_YEAR_FILTER is not None:
    if isinstance(EVENT_YEAR_FILTER, (list, set, tuple)):
        df = df[df["event_year"].isin(EVENT_YEAR_FILTER)].reset_index(drop=True)
    else:
        df = df[df["event_year"] == EVENT_YEAR_FILTER].reset_index(drop=True)
    print(f"Filtered by event_year: {len(df):,} rows remaining")

## 4. Prompt Builder (chat template or plain text for base models)

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant. Answer the question as concisely as "
    "possible, ideally in a few words or a short phrase. "
    "Do not explain your reasoning. Specify the answer only. "
    "Provide an answer after 'Answer:'. "
    "Example: Question: what is the color of the sky?\nAnswer: blue"
)

PLAIN_PROMPT_TEMPLATE = (
    f"{SYSTEM_PROMPT}\n\nQuestion: {{question}}\n\nAnswer:"
)


def get_question_text(row, prompt_type: str) -> str:
    pcfg = PROMPT_TYPES[prompt_type]
    question = str(row[pcfg["question_col"]])
    if "year_col" in pcfg:
        year = row[pcfg["year_col"]]
        return pcfg["template"].format(year=int(year), question=question)
    return question


def build_prompt(question: str, tokenizer) -> str:
    """Build raw prompt string: use chat template if available, else plain text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
            reasoning_effort="low",
        )
    except Exception:
        return PLAIN_PROMPT_TEMPLATE.format(question=question)


print("Prompt builder ready - prompt types:", list(PROMPT_TYPES.keys()))

## 5. Load vLLM Model

In [ ]:
llm_kwargs = {
    "model": MODEL_NAME,
    "trust_remote_code": True,
    "max_model_len": MAX_MODEL_LEN,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "tensor_parallel_size": NUM_GPUS
}
if CACHE_DIR:
    llm_kwargs["download_dir"] = CACHE_DIR

# Load tokenizer for prompt building (use apply_chat_template or plain fallback for base models)
print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR, trust_remote_code=True, fix_mistral_regex=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left padding for decoder-only models (generation from right edge)

print(f"Loading vLLM model: {MODEL_NAME} ...")
llm = LLM(**llm_kwargs)
print("Model loaded.")

# Sampling params: greedy decoding, optional stop at newline
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
    stop=STOP_STRINGS if STOP_STRINGS else None,
)

## 6. Build Prompts for All Types (raw strings for llm.generate)

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build raw prompts (strings) for llm.generate — avoids vLLM chat template requirement for base models
prompts_by_type = {}
keys_by_type = {}

for ptype in PROMPT_TYPES_YEAR_DEPENDENT:
    prompts = []
    for _, row in df.iterrows():
        q = get_question_text(row, ptype)
        prompts.append(build_prompt(q, tokenizer))
    prompts_by_type[ptype] = prompts
    keys_by_type[ptype] = None  # 1:1 with df
    print(f"  [{ptype:>10}] {len(prompts):,} prompts (full dataset)")

# Year-independent: deduplicate by question
for ptype in PROMPT_TYPES_YEAR_INDEPENDENT:
    key_col = PROMPT_TYPES[ptype]["question_col"]
    df_unique = df.drop_duplicates(subset=[key_col]).reset_index(drop=True)
    prompts = []
    keys = []
    for _, row in df_unique.iterrows():
        q = str(row[key_col])
        prompts.append(build_prompt(q, tokenizer))
        keys.append(q)
    prompts_by_type[ptype] = prompts
    keys_by_type[ptype] = keys
    print(f"  [{ptype:>10}] {len(prompts):,} unique prompts (saving {len(df) - len(df_unique):,} duplicate inferences)")

## 7. Batched Inference with vLLM

In [ ]:
def run_inference_chunked(prompts: list, ptype: str, start_offset: int = 0,
                         checkpoint_cb=None) -> tuple[list, float]:
    """Run vLLM generate in chunks. checkpoint_cb(preds_list) called periodically for year-dependent."""
    predictions = []
    wall_start = time.time()
    remaining = prompts[start_offset:]
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc=f"vLLM [{ptype}]", unit="batch"):
        batch = remaining[i : i + BATCH_SIZE]
        outputs = llm.generate(batch, sampling_params, use_tqdm=False)
        for out in outputs:
            raw = out.outputs[0].text if out.outputs else ""
            # Clean up: take first line, then remove "Answer:" prefix if present
            text = raw.strip().split("\n")[0].strip() if raw.strip() else ""
            text = text.split("Answer:")[-1].strip() if text else ""
            predictions.append(text if text else "")
        if checkpoint_cb and len(predictions) > 0 and (start_offset + len(predictions)) // CHECKPOINT_EVERY > (start_offset + len(predictions) - len(batch)) // CHECKPOINT_EVERY:
            checkpoint_cb(predictions)
    return predictions, time.time() - wall_start


all_predictions = {}
wall_times = {}
wall_start_total = time.time()

In [ ]:
dataset_stem = Path(DATASET_PATH).stem
cache_dir = Path(OUTPUT_DIR) / CACHE_SUBDIR
cache_dir.mkdir(parents=True, exist_ok=True)

for ptype in PROMPT_TYPES:
    answer_col = f"model_answer_{ptype}"
    checkpoint_path = Path(OUTPUT_DIR) / f"checkpoint_{safe_model}_{RUN_TAG}_{ptype}.parquet"
    prompts = prompts_by_type[ptype]
    keys = keys_by_type[ptype]
    is_year_independent = ptype in PROMPT_TYPES_YEAR_INDEPENDENT

    if is_year_independent:
        key_col = PROMPT_TYPES[ptype]["question_col"]
        cache_path = cache_dir / f"cache_{ptype}_{safe_model}_{dataset_stem}.parquet"
        mapping = {}

        if USE_INVARIANT_RELATIVE_CACHE and cache_path.exists():
            cache_df = pd.read_parquet(cache_path)
            mapping = dict(zip(cache_df[key_col], cache_df[answer_col]))
            print(f"  [{ptype}] Loaded from cache: {len(mapping):,} unique")

        remaining_keys = [k for k in keys if k not in mapping]
        remaining_prompts = [build_prompt(k, tokenizer) for k in remaining_keys]

        if not remaining_prompts:
            print(f"  [{ptype}] Already complete — skipping")
            wall_times[ptype] = 0.0
            all_predictions[ptype] = ("mapping", mapping)
            continue

        preds, elapsed = run_inference_chunked(remaining_prompts, ptype)
        for k, p in zip(remaining_keys, preds):
            mapping[k] = p

        if USE_INVARIANT_RELATIVE_CACHE:
            pd.DataFrame({key_col: list(mapping.keys()), answer_col: list(mapping.values())}).to_parquet(cache_path, index=False)

        wall_times[ptype] = elapsed
        all_predictions[ptype] = ("mapping", mapping)
    else:
        resume_offset = 0
        preds_so_far = []
        if checkpoint_path.exists():
            ckpt_df = pd.read_parquet(checkpoint_path, columns=[answer_col])
            resume_offset = len(ckpt_df)
            preds_so_far = ckpt_df[answer_col].tolist()
            del ckpt_df
            print(f"  [{ptype}] Resuming from checkpoint: {resume_offset:,} rows done")

        remaining = len(prompts) - resume_offset
        if remaining <= 0:
            print(f"  [{ptype}] Already complete — skipping")
            wall_times[ptype] = 0.0
            all_predictions[ptype] = ("list", preds_so_far)
            continue

        def _save_ckpt(new_preds):
            full_preds = preds_so_far + new_preds
            _df = df.iloc[: len(full_preds)].copy()
            _df[answer_col] = full_preds
            _df.to_parquet(checkpoint_path, index=False)

        preds_new, elapsed = run_inference_chunked(prompts, ptype, resume_offset, checkpoint_cb=_save_ckpt)
        preds = preds_so_far + preds_new
        if checkpoint_path.exists():
            checkpoint_path.unlink()
        wall_times[ptype] = elapsed
        all_predictions[ptype] = ("list", preds)

    n = len(mapping) if is_year_independent else len(preds)
    print(f"  {ptype}: {n:,} predictions in {elapsed / 60:.1f} min ({n / max(elapsed, 0.01):.1f} q/s)")

wall_elapsed = time.time() - wall_start_total
print(f"\nTotal wall time: {wall_elapsed / 60:.1f} min")

## 8. Assemble & Save Results

In [ ]:
for ptype, pred_data in all_predictions.items():
    kind, data = pred_data
    answer_col = f"model_answer_{ptype}"
    if kind == "mapping":
        key_col = PROMPT_TYPES[ptype]["question_col"]
        df[answer_col] = df[key_col].map(data)
    else:
        df[answer_col] = data

df["model_name"] = MODEL_NAME
df["run_tag"] = RUN_TAG
df["timestamp"] = datetime.utcnow().isoformat()

output_path = Path(OUTPUT_DIR) / f"benchmark_{safe_model}_{RUN_TAG}.parquet"
df.to_parquet(output_path, index=False)
print(f"Results saved → {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)")

## 9. Summary & Evaluation

In [ ]:
import re, string, ast
from collections import Counter


def normalise(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def _as_list(val):
    if isinstance(val, list):
        return [str(v) for v in val]
    s = str(val).strip()
    if s.startswith("["):
        try:
            p = ast.literal_eval(s)
            return [str(v) for v in p] if isinstance(p, list) else [s]
        except (ValueError, SyntaxError):
            pass
    return [s]


def exact_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) == pred for t in _as_list(truth)) else 0.0


def token_f1(pred: str, truth) -> float:
    pred_tokens = normalise(pred).split()
    truths = _as_list(truth)
    if not pred_tokens:
        return 0.0
    best = 0.0
    for t in truths:
        tt = normalise(t).split()
        if not tt:
            continue
        common = Counter(pred_tokens) & Counter(tt)
        n = sum(common.values())
        if n == 0:
            continue
        prec = n / len(pred_tokens)
        rec = n / len(tt)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best


def contains_match(pred: str, truth) -> float:
    pred = normalise(pred)
    return 1.0 if any(normalise(t) in pred for t in _as_list(truth)) else 0.0


summary = {
    "model": MODEL_NAME,
    "run_tag": RUN_TAG,
    "backend": "vllm",
    "prompt_types": list(PROMPT_TYPES.keys()),
    "dataset_rows": len(df),
    "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "total_wall_time_min": round(wall_elapsed / 60, 2),
    "output_file": str(output_path),
}

for ptype in PROMPT_TYPES:
    col = f"model_answer_{ptype}"
    preds = df[col].fillna("").astype(str)
    truths = df[ANSWER_COL]
    summary[f"{ptype}_em"] = round(sum(exact_match(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_f1"] = round(sum(token_f1(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_contains"] = round(sum(contains_match(p, t) for p, t in zip(preds, truths)) / len(df), 4)
    summary[f"{ptype}_wall_min"] = round(wall_times[ptype] / 60, 2)

summary_path = Path(OUTPUT_DIR) / f"benchmark_summary_{safe_model}_{RUN_TAG}.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))